# Train the MLBB vision detector (free, in Google Colab)

Makes `mlbb_detector.tflite` + `labels.txt` for the MLBB Research app (Model page → *Choose detector file*).

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (free).
2. Make a free account at roboflow.com and copy your **API key** (Settings → API Keys). You type it into the box below; it is never saved in this notebook.
3. Default dataset: **Mobile Legends Minimap Dataset** by *gladi* on Roboflow Universe, licence **CC BY 4.0** (credit the author if you share the model).

Run each cell from top to bottom (tap ▶). Training takes about 10–20 minutes.

**Retraining keeps what it learned:** each run continues from your last detector (`*_detector_best.pt`, saved in My Drive/MLBB-Research) and uses every picture zip in that folder. Keep the zips and that file in Drive.

In [ ]:
!pip install -q ultralytics roboflow

## 1. Settings
Change these only if you want a different Roboflow dataset. For a full-screen dataset, also set the app's *Detector looks at* to *the whole screen*.

In [ ]:
WORKSPACE = "gladi"                            # Roboflow workspace (from the dataset URL)
PROJECT   = "mobile-legends-minimap-dataset"   # Roboflow project (from the dataset URL)
VERSION   = 2                                  # dataset version number
EPOCHS    = 80                                 # more = better but slower
IMGSZ     = 320                                # 320 is fast on phones; 640 is more accurate

## 2. Download the dataset

In [ ]:
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass("Paste your Roboflow API key and press Enter: "))
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")
print("Downloaded to", dataset.location)

## 2b. Add YOUR OWN pictures (optional, recommended)
The app saves minimap pictures with boxes already drawn while you play (Learn page → *Collect detector training pictures*), in `Download/MLBB-Research/dataset` (`images/`, `labels/`, `classes.txt`).

**Best:** upload that folder to your Roboflow project first, fix the wrong / missing boxes there, and export a new version (then just change `VERSION` above).

**Quick:** zip the `dataset` folder on your phone (file manager → compress), run the next cell and choose the zip. Its pictures are added to the training set as they are.

In [ ]:
import zipfile, glob, os, random, shutil, yaml
# Your files: taken from Google Drive (My Drive/MLBB-Research) when they are there - nothing to upload.
# If that folder has none, you pick them on this device instead. (Google asks once to allow Drive access.)
import glob, os
from google.colab import files
DRIVE_DIR = "/content/drive/MyDrive/MLBB-Research"
paths = []
try:
    from google.colab import drive
    drive.mount("/content/drive")
    for pat in ["mlbb-minimap-dataset-*.zip", "dataset*.zip"]:
        paths += sorted(glob.glob(f"{DRIVE_DIR}/{pat}"))
    paths = sorted(set(paths))
except Exception as e:
    print("Google Drive not used:", e)
if paths:
    print(f"Using {len(paths)} minimap picture zips from Google Drive (My Drive/MLBB-Research):", ", ".join(os.path.basename(p) for p in paths))
else:
    print("No minimap picture zips in My Drive/MLBB-Research - pick them on this device:")
    up = files.upload()
    paths = [os.path.abspath(k) for k in up]
shutil.rmtree("/content/mine", ignore_errors=True)
for k in paths:     # every zip (several batches are fine: no picture is in two zips)
    if k.endswith(".zip"): zipfile.ZipFile(k).extractall("/content/mine/" + os.path.basename(k)[:-4])
imgs = sorted(glob.glob("/content/mine/**/images/*.png", recursive=True))
print(len(imgs), "of your pictures")

# your class names must match the Roboflow dataset's
rf_yaml = yaml.safe_load(open(f"{dataset.location}/data.yaml"))
names = rf_yaml["names"]
cls_files = glob.glob("/content/mine/**/classes.txt", recursive=True)
mine_names = [l.strip() for l in open(cls_files[0]) if l.strip()] if cls_files else list(names)
# Your boxes are numbered by the app's class list; the dataset may have other classes / another
# order. Each box is converted by its NAME. Lord and turtle are ADDED as new classes when the
# dataset does not have them (your pictures teach them); other unknown classes are dropped.
names = list(names)
for extra in ("lord_icon", "turtle_icon"):
    if extra not in names: names.append(extra)
remap = {i: names.index(n) for i, n in enumerate(mine_names) if n in names}
print("Classes to train:", names)
print("Your boxes converted by name:", {mine_names[i]: names[j] for i, j in remap.items()},
      "| dropped:", [n for n in mine_names if n not in names])

# Pictures saved before the app knew the lord / turtle icons: box them now. Both pits are at fixed
# spots in the picture; the LORD is a trophy with a thin stem (pink early, gold when up), the
# TURTLE a pink blob. Same rules as the app (MinimapAnalyzer.pits).
import numpy as np
from PIL import Image
LORD, TURTLE = names.index("lord_icon"), names.index("turtle_icon")
PITS = [(71 / 152, 35 / 132), (114 / 152, 80 / 132)]     # pit centres as fractions of the minimap picture
def pit_boxes(path):
    im = np.asarray(Image.open(path).convert("RGB"), dtype=int); H, W = im.shape[:2]
    out = []
    for px_, py_ in PITS:
        cx, cy = int(px_ * W), int(py_ * H); hw, hh = int(W * 0.1), int(H * 0.11)
        win = im[max(0, cy - hh):cy + hh, max(0, cx - hw):cx + hw]; oy, ox = max(0, cy - hh), max(0, cx - hw)
        r, g, b = win[..., 0], win[..., 1], win[..., 2]
        gold = (r > 170) & (g > 130) & (b < 125) & (r - b > 70) & (r >= g)
        mag = (r > 170) & (b > 140) & (g < 140) & (r - g > 60) & (b - g > 60)
        m = gold | mag
        if m.sum() < 10: continue
        ys, xs = np.nonzero(m)
        # the biggest group of pixels (2 px apart across, 3 up-down count as one shape)
        pts = list(zip(ys, xs)); seen = set(); best = []
        for p0 in pts:
            if p0 in seen: continue
            grp, st = [], [p0]; seen.add(p0)
            while st:
                y, x = st.pop(); grp.append((y, x))
                for q in pts:
                    if q not in seen and abs(q[0] - y) <= 3 and abs(q[1] - x) <= 2: seen.add(q); st.append(q)
            if len(grp) > len(best): best = grp
        if len(best) < 10: continue
        ys = np.array([p[0] for p in best]); xs = np.array([p[1] for p in best])
        x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
        if x1 - x0 > W * 0.21 or y1 - y0 > H * 0.22: continue
        golden = sum(gold[y, x] for y, x in best) * 2 > len(best)
        mid = (x0 + x1) / 2; run = 0; stem = False
        for yy in range(y0 + (y1 - y0) // 3, y1 + 1):
            row = xs[ys == yy]
            run = run + 1 if 1 <= len(row) <= 2 and abs(row.mean() - mid) <= 2.5 else 0
            stem = stem or run >= 3
        cls = LORD if golden or stem else TURTLE
        out.append(f"{cls} {(ox + (x0 + x1) / 2) / W:.5f} {(oy + (y0 + y1) / 2) / H:.5f} {(x1 - x0 + 3) / W:.5f} {(y1 - y0 + 3) / H:.5f}")
    return out

pit_added = [0]
# 85% of your pictures go to training, 15% to checking
random.seed(0); random.shuffle(imgs)
for part, lst in (("train", imgs[:int(len(imgs) * 0.85)]), ("valid", imgs[int(len(imgs) * 0.85):])):
    os.makedirs(f"/content/mine_{part}/images", exist_ok=True); os.makedirs(f"/content/mine_{part}/labels", exist_ok=True)
    for im in lst:
        lb = im.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        if not os.path.exists(lb): continue
        rows = []
        for line in open(lb):
            v = line.split()
            if len(v) == 5 and int(v[0]) in remap: rows.append(" ".join([str(remap[int(v[0])])] + v[1:]))
        if not any(r.split()[0] in (str(LORD), str(TURTLE)) for r in rows):
            added = pit_boxes(im); rows += added; pit_added[0] += len(added)
        shutil.copy(im, f"/content/mine_{part}/images/")
        open(f"/content/mine_{part}/labels/" + os.path.basename(lb), "w").write("\n".join(rows) + ("\n" if rows else ""))

print("lord / turtle boxes added to your older pictures:", pit_added[0])

def rf_dir(k):
    v = rf_yaml[k]
    return v if os.path.isabs(v) else os.path.normpath(os.path.join(dataset.location, v.replace("../", "")))
combined = {"train": [rf_dir("train"), "/content/mine_train/images"],
            "val": [rf_dir("val"), "/content/mine_valid/images"],
            "nc": len(names), "names": names}
yaml.safe_dump(combined, open("/content/combined.yaml", "w"))
DATA = "/content/combined.yaml"
print("Training on the Roboflow set + your pictures:", DATA)

## 3. Train a small YOLOv8 model

In [ ]:
from ultralytics import YOLO

# CONTINUES from your last trained detector when it is in My Drive/MLBB-Research (minimap_detector_best.pt):
# it keeps what it learned and gets better with the new pictures (old zips in Drive are used again too).
# The first time (no file yet) it starts from the general YOLO 'nano' model (small and fast, good for phones).
import os, shutil
PREV = "/content/drive/MyDrive/MLBB-Research/minimap_detector_best.pt"
if os.path.exists(PREV):
    print("Continuing from your last trained detector:", PREV)
    model = YOLO(PREV)
else:
    print("No earlier detector in My Drive/MLBB-Research: starting fresh")
    model = YOLO("yolov8n.pt")
DATA = globals().get("DATA", f"{dataset.location}/data.yaml")   # your pictures too, if added above
model.train(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=16, patience=20)
best_path = model.trainer.best
print("Best weights:", best_path)
# keep it for next time (the next run continues from it)
shutil.copy(best_path, "/content/minimap_detector_best.pt")
try:
    if os.path.isdir("/content/drive/MyDrive"):
        os.makedirs("/content/drive/MyDrive/MLBB-Research", exist_ok=True)
        shutil.copy(best_path, PREV)
        print("Saved for next time: My Drive/MLBB-Research/minimap_detector_best.pt")
except Exception as e:
    print("(could not save it to Drive:", e, ") - download minimap_detector_best.pt and put it in My Drive/MLBB-Research")

## 4. Export to LiteRT (.tflite) and download

In [ ]:
import glob, os, shutil, yaml

best = YOLO(best_path)
# Newer Ultralytics calls this format "litert"; older versions call it "tflite"
try:
    out = best.export(format="litert", imgsz=IMGSZ)
except Exception as e:
    print("litert export not available here, using tflite:", e)
    out = best.export(format="tflite", imgsz=IMGSZ)
print("Export result:", out)

# find the .tflite wherever Ultralytics saved it
weights_dir = os.path.dirname(str(best_path))
found = []
if out and str(out).endswith(".tflite"):
    found.append(str(out))
found += sorted(glob.glob(weights_dir + "/**/*.tflite", recursive=True))
found += sorted(glob.glob("/content/**/*.tflite", recursive=True))
found = [f for f in found if os.path.exists(f) and not f.endswith("mlbb_detector.tflite")]
if not found:
    print("No .tflite found. Files next to best.pt:")
    for root, dirs, fs in os.walk(weights_dir):
        for f in fs:
            print(os.path.join(root, f))
    raise SystemExit("Send this list to Claude.")
float32 = [f for f in found if "float32" in f]
pick = (float32 or found)[0]
shutil.copy(pick, "/content/mlbb_detector.tflite")
print("Using:", pick, round(os.path.getsize(pick) / 1e6, 1), "MB")

names = best.names   # the classes the model was trained on (incl. lord / turtle added from your pictures)
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
open("/content/labels.txt", "w").write(",".join(names))
print("Classes (paste into the app if they differ from the default):")
print(",".join(names))

## 5. Download the files
On a phone the direct download often does nothing. If so, run the **Google Drive** cell instead: the files then appear in your Google Drive (My Drive), and the Drive app can save them to the phone.

In [ ]:
import shutil, os
# A. Direct download (works best on a computer)
from google.colab import files
files.download("/content/mlbb_detector.tflite")
files.download("/content/labels.txt")
try:   # also keep a copy in My Drive/MLBB-Research when Google Drive is connected
    if os.path.isdir("/content/drive/MyDrive"):
        os.makedirs("/content/drive/MyDrive/MLBB-Research", exist_ok=True)
        shutil.copy("/content/mlbb_detector.tflite", "/content/drive/MyDrive/MLBB-Research/")
        shutil.copy("/content/labels.txt", "/content/drive/MyDrive/MLBB-Research/")
        print("Also saved to Google Drive > My Drive/MLBB-Research")
except Exception as e: print("(not copied to Drive:", e, ")")

In [ ]:
# B. Save to Google Drive (most reliable on a phone; asks you to allow Drive access)
from google.colab import drive
drive.mount("/content/drive")
shutil.copy("/content/mlbb_detector.tflite", "/content/drive/MyDrive/mlbb_detector.tflite")
shutil.copy("/content/labels.txt", "/content/drive/MyDrive/mlbb_labels.txt")
print("Saved to Google Drive > My Drive: mlbb_detector.tflite and mlbb_labels.txt")

## 6. Put it in the app
1. Copy `mlbb_detector.tflite` to your phone (it downloads straight to the phone if you run Colab there).
2. App → **Model** page → *Choose detector file (.tflite)*.
3. If the class list printed above differs from the app's default, paste it into *Class names*.
4. Home page preview shows the detections as cyan boxes.